# Ekstrakcija ključnih tačaka iz videa (ASL Citizen)

Ovaj notebook se pokreće JEDNOM. Pretvara video snimke znakova u nizove
ključnih tačaka koje koristi model.

Tok:
1. Preuzima se ASL Citizen dataset (Microsoft Research) i raspakuje se podskup
   od 25 izabranih znakova (973 videa: 497 train, 96 val, 380 test).
2. Svaki video se propušta kroz MediaPipe (HandLandmarker + PoseLandmarker),
   frejm po frejm. Iz svakog frejma se izvlači:
   - leva šaka: 21 tačka × 3 (x,y,z) = 63
   - desna šaka: 21 tačka × 3 = 63
   - poza (ramena, laktovi, zglobovi): 12 tačaka × 3 = 36
   Ukupno 162 vrednosti po frejmu. Lice se izostavlja jer ne nosi informaciju
   za izolovane znakove.
3. Rezultat svakog videa (niz [broj_frejmova × 162]) snima se kao .npy fajl
   na Google Drive. Ti .npy fajlovi su ulaz za notebook sa obukom.

Napomena: ekstrakcija je spora (MediaPipe radi frejm po frejm na CPU) i radi
se samo jednom. Dalji rad (priprema, modeli, obuka) koristi gotove .npy tačke.

## Ekstrakcija: video → ključne tačke

Za svaki video, MediaPipe detektuje položaj šaka i tela u svakom frejmu i
vraća koordinate tačaka. Tačke koje nisu detektovane (npr. šaka van kadra)
popunjavaju se nulama. Sve sekvence se snimaju kao .npy na Drive, uz preskakanje
već obrađenih (da se ekstrakcija može nastaviti ako se sesija prekine).

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))

True


In [ ]:
!wget -q --show-progress \
  https://download.microsoft.com/download/b/8/8/b88c0bae-e6c1-43e1-8726-98cf5af36ca4/ASL_Citizen.zip \
  -O /content/ASL_Citizen.zip
print("Skinuto.")

/content/ASL_Citize 100%[===================>]  42.77G   109MB/s    in 9m 3s   
Skinuto.


In [ ]:
import zipfile
z = zipfile.ZipFile("/content/ASL_Citizen.zip")
names = z.namelist()
print("Ukupno stavki:", len(names))
for n in names[:20]:
    print(n)
csvs = [n for n in names if n.endswith(".csv")]
print("\nCSV fajlovi:")
for c in csvs:
    print(" ", c)

Ukupno stavki: 83406
ASL_Citizen/
ASL_Citizen/splits/
ASL_Citizen/splits/val.csv
ASL_Citizen/splits/train.csv
ASL_Citizen/splits/test.csv
ASL_Citizen/videos/
ASL_Citizen/videos/5086723486875651-RUN.mp4
ASL_Citizen/videos/2233415645150214-SWEET.mp4
ASL_Citizen/videos/09412352698319548-POSSIBLE.mp4
ASL_Citizen/videos/6184880487426632-PAINT.mp4
ASL_Citizen/videos/6682045893304822-RACE.mp4
ASL_Citizen/videos/22425009794174322-SCULPT.mp4
ASL_Citizen/videos/8563900568328626-SCAR MEMORY.mp4
ASL_Citizen/videos/6843553981288921-EARTHQUAKE.mp4
ASL_Citizen/videos/37687322393253075-REPLACE.mp4
ASL_Citizen/videos/8136380896447564-WRISTWATCH 3.mp4
ASL_Citizen/videos/023749434963018734-WHISTLE.mp4
ASL_Citizen/videos/639898672430512-PARTY 2.mp4
ASL_Citizen/videos/2743155783934501-ELEGANCE.mp4
ASL_Citizen/videos/11137281856386338-MENTION.mp4

CSV fajlovi:
  ASL_Citizen/splits/val.csv
  ASL_Citizen/splits/train.csv
  ASL_Citizen/splits/test.csv


In [ ]:
import zipfile, pandas as pd, os

WORK = "/content/drive/MyDrive/asl_citizen"
os.makedirs(f"{WORK}/splits", exist_ok=True)

z = zipfile.ZipFile("/content/ASL_Citizen.zip")
for split in ["train.csv", "val.csv", "test.csv"]:
    src = f"ASL_Citizen/splits/{split}"
    with z.open(src) as f, open(f"{WORK}/splits/{split}", "wb") as out:
        out.write(f.read())
print("CSV-ovi sačuvani na Drive.")

train = pd.read_csv(f"{WORK}/splits/train.csv")
print("\nKolone:", list(train.columns))
print("Redova u train:", len(train))
print(train.head())

CSV-ovi sačuvani na Drive.

Kolone: ['Participant ID', 'Video file', 'Gloss', 'ASL-LEX Code']
Redova u train: 40154
  Participant ID                        Video file       Gloss ASL-LEX Code
0             P1       15890366051589533-APPLE.mp4       APPLE     A_03_054
1             P1  35618482303951104-IMPOSSIBLE.mp4  IMPOSSIBLE     B_01_032
2             P1         6958143575951994-PARK.mp4        PARK     E_03_028
3             P1     8006032738002744-SOCCER 2.mp4     SOCCER2     F_03_032
4             P1       37542279833186454-STINK.mp4       STINK     H_01_064


##  Analiza skupa i izbor 25 znakova
Broj primera po znaku i izbor podskupa od 25 znakova sa najviše primera,
koji postoje u sva tri podskupa (trening, validacija, test).

In [ ]:
gloss_col = [c for c in train.columns if 'gloss' in c.lower()][0]
print("Kolona sa znakom:", gloss_col)
print("Broj različitih znakova:", train[gloss_col].nunique())

vc = train[gloss_col].value_counts()
print("Primera po znaku — min:", vc.min(), "max:", vc.max(), "mean:", round(vc.mean(),1))
print("\nPrimer nekoliko znakova:")
print(vc.head(15))

Kolona sa znakom: Gloss
Broj različitih znakova: 2731
Primera po znaku — min: 9 max: 24 mean: 14.7

Primer nekoliko znakova:
Gloss
DOG1             24
HURDLE/TRIP1     22
DEMAND1          21
BITE1            21
DARK1            21
BREAKFAST1       21
MECHANIC1        20
PARTY1           20
ROCKINGCHAIR1    20
DEAF1            20
WHATFOR1         20
DECIDE1          20
FINE1            19
NOON1            19
HALLOWEEN1       19
Name: count, dtype: int64


In [ ]:
import pandas as pd

WORK = "/content/drive/MyDrive/asl_citizen"
train = pd.read_csv(f"{WORK}/splits/train.csv")
val   = pd.read_csv(f"{WORK}/splits/val.csv")
test  = pd.read_csv(f"{WORK}/splits/test.csv")

# Znakovi koji postoje u sva tri splita
common = set(train.Gloss) & set(val.Gloss) & set(test.Gloss)
print("Znakova u sva tri splita:", len(common))

# Medju njima, biramo onih 25 sa najvise primera u trainu
counts = train[train.Gloss.isin(common)].Gloss.value_counts()
SELECTED = counts.head(25).index.tolist()

print("\nIzabrani znakovi (25):")
for s in SELECTED:
    n_tr = (train.Gloss == s).sum()
    n_va = (val.Gloss == s).sum()
    n_te = (test.Gloss == s).sum()
    print(f"  {s:18s} train={n_tr:2d}  val={n_va}  test={n_te}")

# Sačuvaj listu da je koristimo kasnije
import json
with open(f"{WORK}/selected_signs.json", "w") as f:
    json.dump(SELECTED, f)
print("\nLista sačuvana u selected_signs.json")

Znakova u sva tri splita: 2731

Izabrani znakovi (25):
  DOG1               train=24  val=4  test=17
  HURDLE/TRIP1       train=22  val=3  test=13
  DEMAND1            train=21  val=4  test=14
  BITE1              train=21  val=4  test=14
  DARK1              train=21  val=3  test=15
  BREAKFAST1         train=21  val=3  test=15
  MECHANIC1          train=20  val=3  test=16
  PARTY1             train=20  val=4  test=15
  ROCKINGCHAIR1      train=20  val=4  test=15
  DEAF1              train=20  val=4  test=15
  WHATFOR1           train=20  val=5  test=18
  DECIDE1            train=20  val=3  test=15
  FINE1              train=19  val=4  test=16
  NOON1              train=19  val=4  test=15
  HALLOWEEN1         train=19  val=4  test=14
  DEVELOP1           train=19  val=3  test=14
  LUNCH1             train=19  val=5  test=14
  EDIT1              train=19  val=4  test=15
  SHAVE1             train=19  val=5  test=15
  BACKPACK1          train=19  val=3  test=15
  BEE1               trai

## Raspakivanje izabranih videa
Iz arhiva se raspakuju samo video snimci izabranih 25 znakova (973 snimka),
ne svih 84.000.

In [ ]:
import zipfile, pandas as pd, json, os

WORK = "/content/drive/MyDrive/asl_citizen"
with open(f"{WORK}/selected_signs.json") as f:
    SELECTED = json.load(f)

train = pd.read_csv(f"{WORK}/splits/train.csv")
val   = pd.read_csv(f"{WORK}/splits/val.csv")
test  = pd.read_csv(f"{WORK}/splits/test.csv")

# Skupi imena svih video fajlova koji pripadaju izabranim znakovima
wanted = {}  # split -> lista video fajlova
for name, df in [("train", train), ("val", val), ("test", test)]:
    files = df[df.Gloss.isin(SELECTED)]["Video file"].tolist()
    wanted[name] = files
    print(f"{name}: {len(files)} videa")

total = sum(len(v) for v in wanted.values())
print("Ukupno videa za raspakivanje:", total)

# Raspakuj ih na Drive, organizovano po splitu
z = zipfile.ZipFile("/content/ASL_Citizen.zip")
for split, files in wanted.items():
    out_dir = f"{WORK}/videos/{split}"
    os.makedirs(out_dir, exist_ok=True)
    for vf in files:
        src = f"ASL_Citizen/videos/{vf}"
        try:
            with z.open(src) as fsrc, open(f"{out_dir}/{vf}", "wb") as fout:
                fout.write(fsrc.read())
        except KeyError:
            print("  Nedostaje u zipu:", vf)
    print(f"{split}: raspakovano u {out_dir}")

print("Gotovo.")

train: 497 videa
val: 96 videa
test: 380 videa
Ukupno videa za raspakivanje: 973
train: raspakovano u /content/drive/MyDrive/asl_citizen/videos/train
val: raspakovano u /content/drive/MyDrive/asl_citizen/videos/val
test: raspakovano u /content/drive/MyDrive/asl_citizen/videos/test
Gotovo.


## MediaPipe — instalacija i modeli
Instalira se MediaPipe i preuzimaju gotovi modeli za detekciju šaka (HandLandmarker)
i tela (PoseLandmarker).

In [ ]:
!pip uninstall -y mediapipe protobuf -q
!pip install -q --upgrade mediapipe
print("Instalirano. OBAVEZNO: Runtime -> Restart session, pa kreni od Ćelije 8b.")

Instalirano. OBAVEZNO: Runtime -> Restart session, pa kreni od Ćelije 8b.


In [ ]:
!wget -q -O /content/hand_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -O /content/pose_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
print("Modeli skinuti.")

Modeli skinuti.


## Definicija ključnih tačaka
Određuje se koje tačke se izdvajaju: 21 tačka po šaci (dve šake) i 12 tačaka poze
(ramena, laktovi, zglobovi). Svaka tačka ima tri koordinate (x, y, z),
pa jedan frejm opisuje 162 vrednosti. Lice se izostavlja jer za pojedinačne znakove
ne nosi značajnu informaciju.

In [ ]:
import numpy as np

N_HAND = 21
# Gornji deo tela: ramena, laktovi, zglobovi (poza ima 33 tačke; uzimamo 11-22)
POSE_IDX = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
N_POSE = len(POSE_IDX)

# 2 šake * 21 * 3  +  12 tačaka poze * 3
FEATURES_PER_FRAME = N_HAND*3*2 + N_POSE*3   # 126 + 36 = 162
print("Karakteristika po frejmu:", FEATURES_PER_FRAME)

Karakteristika po frejmu: 162


## Ekstrakcija ključnih tačaka iz videa
Glavna funkcija: otvara video, obrađuje ga frejm po frejm kroz MediaPipe i za svaki
frejm izdvaja položaj obe šake i poze. Tačke koje nisu detektovane popunjavaju se
nulama. Rezultat je matrica [broj_frejmova × 162].

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

def make_landmarkers():
    hand_opts = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="/content/hand_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    pose_opts = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="/content/pose_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    hand = vision.HandLandmarker.create_from_options(hand_opts)
    pose = vision.PoseLandmarker.create_from_options(pose_opts)
    return hand, pose

def extract_keypoints_from_video(video_path):
    """Vrati [T, 162]: leva šaka(63) + desna šaka(63) + poza(36). Fali -> nule."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    hand_lm, pose_lm = make_landmarkers()
    frames = []
    idx = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts = int(idx * 1000 / fps)

        # ŠAKE
        left = np.zeros(N_HAND*3)
        right = np.zeros(N_HAND*3)
        hres = hand_lm.detect_for_video(mp_image, ts)
        if hres.hand_landmarks:
            for hand_pts, handed in zip(hres.hand_landmarks, hres.handedness):
                pts = np.array([[p.x, p.y, p.z] for p in hand_pts]).flatten()
                if handed[0].category_name == "Left":
                    left = pts
                else:
                    right = pts

        # POZA
        pose = np.zeros(N_POSE*3)
        pres = pose_lm.detect_for_video(mp_image, ts)
        if pres.pose_landmarks:
            lms = pres.pose_landmarks[0]
            pose = np.array([[lms[i].x, lms[i].y, lms[i].z] for i in POSE_IDX]).flatten()

        frames.append(np.concatenate([left, right, pose]))
        idx += 1
    cap.release()
    hand_lm.close()
    pose_lm.close()
    return np.array(frames, dtype=np.float32)

# Test na jednom videu
import glob
test_video = glob.glob("/content/drive/MyDrive/asl_citizen/videos/train/*.mp4")[0]
kp = extract_keypoints_from_video(test_video)
print("Test video:", test_video.split("/")[-1])
print("Oblik izlaza [frejmova, karakteristika]:", kp.shape)

Test video: 4520498201410337-BACKPACK.mp4
Oblik izlaza [frejmova, karakteristika]: (76, 162)


In [ ]:
import glob, os, time
import numpy as np

WORK = "/content/drive/MyDrive/asl_citizen"

for split in ["train", "val", "test"]:
    in_dir = f"{WORK}/videos/{split}"
    out_dir = f"{WORK}/keypoints/{split}"
    os.makedirs(out_dir, exist_ok=True)

    videos = sorted(glob.glob(f"{in_dir}/*.mp4"))
    print(f"\n=== {split}: {len(videos)} videa ===")
    t0 = time.time()

    for i, vp in enumerate(videos):
        name = os.path.splitext(os.path.basename(vp))[0]
        out_path = f"{out_dir}/{name}.npy"

        # Preskoči ako je već urađeno (da možeš da nastaviš ako sesija pukne)
        if os.path.exists(out_path):
            continue

        try:
            kp = extract_keypoints_from_video(vp)
            np.save(out_path, kp)
        except Exception as e:
            print(f"  GREŠKA na {name}: {e}")

        if (i+1) % 50 == 0:
            elapsed = time.time() - t0
            print(f"  {i+1}/{len(videos)}  ({elapsed:.0f}s)")

    print(f"{split} gotov za {time.time()-t0:.0f}s")

print("\nSVE GOTOVO.")


=== train: 497 videa ===
  250/497  (49s)
  300/497  (294s)
  350/497  (513s)
  400/497  (749s)
  450/497  (969s)
train gotov za 1222s

=== val: 96 videa ===
  50/96  (276s)
val gotov za 550s

=== test: 380 videa ===
  50/380  (236s)
  100/380  (422s)
  150/380  (636s)
  200/380  (842s)
  250/380  (1057s)
  300/380  (1287s)
  350/380  (1531s)
test gotov za 1654s

SVE GOTOVO.


##  Pakovanje rezultata
Ključne tačke, podela i lista znakova pakuju se u jedan arhiv radi lakšeg čuvanja
i prenosa. Ovo su ulazni podaci za notebook sa obukom.

In [ ]:
import os, glob, shutil, json
WORK = "/content/drive/MyDrive/asl_citizen"
# 1. Provera da je ekstrakcija stvarno gotova
for split in ["train", "val", "test"]:
    n = len(glob.glob(f"{WORK}/keypoints/{split}/*.npy"))
    print(f"keypoints/{split}: {n} .npy")
# 2. Spakuj sve što treba za dalje u jedan folder
bundle = "/content/asl_bundle"
os.makedirs(bundle, exist_ok=True)
shutil.copytree(f"{WORK}/keypoints", f"{bundle}/keypoints", dirs_exist_ok=True)
shutil.copytree(f"{WORK}/splits", f"{bundle}/splits", dirs_exist_ok=True)
shutil.copy(f"{WORK}/selected_signs.json", f"{bundle}/selected_signs.json")
# 3. Napravi jedan zip i sačuvaj ga na Drive
shutil.make_archive(f"{WORK}/asl_keypoints_bundle", "zip", bundle)
size = os.path.getsize(f"{WORK}/asl_keypoints_bundle.zip")/1e6
print(f"\nSačuvano: {WORK}/asl_keypoints_bundle.zip  ({size:.0f} MB)")

keypoints/train: 497 .npy
keypoints/val: 96 .npy
keypoints/test: 380 .npy

Sačuvano: /content/drive/MyDrive/asl_citizen/asl_keypoints_bundle.zip  (23 MB)
